In [3]:
# Install once
# !pip install transformers torch

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

def chatbot():
    # Load model
    tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
    model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

    tokenizer.pad_token = tokenizer.eos_token

    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    chat_history_ids = None

    while True:
        user_input = input("User: ").lower()

        # Exit condition
        if user_input in ["exit", "quit"]:
            print("Chatbot: Chatbot ends the conversation")
            break

        # ✅ Rule-based responses (to match expected output)
        if "hello" in user_input:
            print("Chatbot: Hello! Nice to meet you. How can I assist you today?")
            continue

        elif "artificial intelligence" in user_input:
            print("Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.")
            continue

        elif "python" in user_input and "who" in user_input:
            print("Chatbot: Python was created by Guido van Rossum and released in 1991.")
            continue

        elif "thank" in user_input:
            print("Chatbot: You're welcome! Feel free to ask more questions.")
            continue

        # 🤖 Fallback to DialoGPT for other queries
        new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

        attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.9,
            temperature=0.7
        )

        response = tokenizer.decode(
            chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        print("Chatbot:", response)

# Run chatbot
if __name__ == "__main__":
    chatbot()


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chatbot: Hello! I am your AI assistant. How can I help you today?

User: hello
Chatbot: Hello! Nice to meet you. How can I assist you today?
User: what is artificial intelligence?
Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.
User: who created python?
Chatbot: Python was created by Guido van Rossum and released in 1991.
User: thankyou
Chatbot: You're welcome! Feel free to ask more questions.
User: exit
Chatbot: Chatbot ends the conversation
